# Provenance

- Cohort: Original experimental cohort.
- Purpose: Construct the aspect-level DORIS-MAE reranking training dataset.
- Matching configuration: `configs/experiments/primary.yaml`.
- External inputs: `external_data/doris_mae/DORIS-MAE_dataset_v1.json`.
- Expected output type: An aspect-level Hugging Face dataset and JSONL training instances under `results/training_data/`.

All paths below are expressed through repository-relative variables; provide external inputs locally without committing them.


In [ ]:
from pathlib import Path

REPOSITORY_ROOT = Path.cwd().resolve()
EXTERNAL_DATA_DIR = REPOSITORY_ROOT / "external_data"
RESULTS_DIR = REPOSITORY_ROOT / "results"
DORIS_MAE_DIR = EXTERNAL_DATA_DIR / "doris_mae"
SPLITS_DIR = RESULTS_DIR / "training_data" / "splits"
TRAINING_DATA_DIR = RESULTS_DIR / "training_data"


In [ ]:
import pandas as pd
import json, itertools
import numpy as np
import ijson
from datasets import Dataset

In [ ]:
INPUT_FILE = DORIS_MAE_DIR / "DORIS-MAE_dataset_v1.json"
SPLITS_DIR.mkdir(parents=True, exist_ok=True)
TRAINING_DATA_DIR.mkdir(parents=True, exist_ok=True)

# Define the top-level keys you want to split
keys_to_split = [
    "Annotation",
    "Corpus",
    "Query",
    "Test_set",
    "aspect2aspect_id",
    "aspect_id2aspect"
]

for key in keys_to_split:
    print(f"⏳ Extracting: {key} ...")
    with open(INPUT_FILE, 'r') as f:
        # Stream items for arrays, or full object if dict
        parser = ijson.items(f, f"{key}")
        data = next(parser)

    # Save to a separate file
    output_file = SPLITS_DIR / f"{key}.json"
    with open(output_file, 'w') as out:
        json.dump(data, out)

    print(f"✅ Saved {key} to {output_file}")


In [ ]:
annotations_df = pd.read_json(SPLITS_DIR / "Annotation.json")

In [ ]:
annotations_df.head()

In [ ]:
query_df = pd.read_json(SPLITS_DIR / "Query.json")

In [ ]:
query_df.head()

In [ ]:
corpus_df = pd.read_json(SPLITS_DIR / "Corpus.json")

In [ ]:
corpus_df.head()

In [ ]:
with open(SPLITS_DIR / "aspect_id2aspect.json", 'r') as f:
        aspect_id2aspect_dict = json.load(f)

In [ ]:
print(aspect_id2aspect_dict)
print(len(aspect_id2aspect_dict))

In [ ]:
print(annotations_df.info())
print(query_df.info())
print(corpus_df.info())

In [ ]:
annotations_df.head()

In [ ]:
import pandas as pd
import numpy as np

def construct_aspect_reranking_dataset(query_df, annotations_df, aspect_id2aspect_dict):
    """
    Constructs an aspect-centric reranking dataset.
    
    """
    try:
        annotations_df['abstract_id'] = annotations_df['abstract_id'].astype(int)
        annotations_df['aspect_id'] = annotations_df['aspect_id'].astype(str)
    except Exception as e:
        print(f"Error in dtype conversion: {e}")
        print("This might be the cause of the 'all zeros' issue.")
    
    print("Pre-processing annotations for fast lookup...")
    # Create a fast lookup for scores: (abstract_id, aspect_id) -> score
    annotation_lookup = annotations_df.set_index(
        ['abstract_id', 'aspect_id']
    )['score'].to_dict()
    print("Annotation lookup created.")
    
    reranking_data = []

    print(f"Processing {len(query_df)} original queries to create aspect-queries...")
    
    # 1. Iterate over each ORIGINAL query
    for _, query in query_df.iterrows():
        
        # --- Get data from the original query ---
        original_candidates = query['candidate_pool']
        main_to_sub_aspect_map = query['aspects']
        
        # ==========================================================
        #  Build the text map from 'aspect_id2sent'
        # ==========================================================
        main_aspect_text_map = {}
        aspect_id_to_sentence_map = query.get('aspect_id2sent', {}) # Get the dict
        
        for aspect_id, sentences in aspect_id_to_sentence_map.items():
            # Join sentences as requested (e.g., "sent1. sent2")
            if sentences: # Ensure list is not empty
                main_aspect_text_map[aspect_id] = ". ".join(sentences)
        # ==========================================================
            
        # --- Get all unique aspect and sub-aspect IDs for this query ---
        all_aspects_in_query = set()
        main_aspect_ids = set(main_to_sub_aspect_map.keys())
        
        for main_aspect, sub_list in main_to_sub_aspect_map.items():
            all_aspects_in_query.add(main_aspect)
            all_aspects_in_query.update(sub_list)

        # 2. Iterate over EVERY aspect/sub-aspect to create a new row
        for aspect_id in all_aspects_in_query:
            
            # --- Requirement 4: Get the query text ---
            query_text = None
            if aspect_id in main_aspect_ids:
                # It's a main aspect, get text from the 'aspect_id2sent' map
                query_text = main_aspect_text_map.get(aspect_id)
            else:
                # It's a sub-aspect, get text from the global dict
                query_text = aspect_id2aspect_dict.get(aspect_id)

            # Skip if we couldn't find text for this aspect
            if query_text is None:
                print(f"Warning: No text found for aspect_id {aspect_id}. Skipping.")
                continue

            # --- Requirement 2: Get the candidate pool ---
            candidate_pool = original_candidates
            
            # --- Requirement 3: Get the candidate scores (as a LIST) ---
            candidate_scores = []
            for abstract_id in candidate_pool:
                score = annotation_lookup.get((abstract_id, aspect_id), 0)
                candidate_scores.append(int(score))

            # 3. Add the new row to our dataset
            reranking_data.append({
                "query_text": query_text,
                "aspect_id": aspect_id,
                "original_query_type": query['query_type'],
                "candidate_pool": candidate_pool,
                "candidate_scores": candidate_scores 
            })

    print(f"Created new reranking DataFrame with {len(reranking_data)} aspect-queries.")
    
    return pd.DataFrame(reranking_data)

In [ ]:
aspect_reranking_df = construct_aspect_reranking_dataset(
    query_df, 
    annotations_df, 
    aspect_id2aspect_dict
)

In [ ]:
aspect_reranking_df.head()

In [ ]:
print(aspect_reranking_df["candidate_scores"][0])

In [ ]:
len(aspect_reranking_df)

In [ ]:
hf_dataset = Dataset.from_pandas(aspect_reranking_df)

hf_dataset.save_to_disk(str(TRAINING_DATA_DIR / "reranking_dataset_doris_mae"))

In [ ]:
aspect_reranking_df.to_json(
    TRAINING_DATA_DIR / "reranking_dataset_doris_mae.jsonl", 
    orient="records", 
    lines=True
)

In [ ]:
print(aspect_reranking_df.loc[aspect_reranking_df["aspect_id"] == "957", "candidate_scores"].iloc[0])
